In [1]:
# =========================================
# CHILD HOPPING TEST (DATA-DRIVEN)
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
from ultralytics import YOLO

In [2]:
# -------------------------------
# MediaPipe Setup
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils
DRAW_LM = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAW_CONN = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# -------------------------------
# Utils
# -------------------------------
def safe_mean(x): return float(np.mean(x)) if len(x) else 0.0
def safe_std(x): return float(np.std(x)) if len(x) else 0.0

def smooth(x, k=5):
    if len(x) < k: return np.array(x)
    return np.convolve(x, np.ones(k)/k, mode='same')

def angle(a,b,c):
    a,b,c = np.array(a),np.array(b),np.array(c)
    ba, bc = a-b, c-b
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

In [4]:
# Same Foot Landing
def foot_score_cal(same_count):
    foot_score = 0
    if same_count >= 5: foot_score = 5
    elif same_count == 4: foot_score = 4
    elif same_count == 3: foot_score = 3
    elif same_count == 2: foot_score = 2
    else: foot_score = 1
    return foot_score

In [5]:
# Free Leg Swing
def swing_score_cal(free_leg_motion):
    swing_score = 0
    if free_leg_motion > 0.05: swing_score = 5
    elif free_leg_motion > 0.04: swing_score = 4
    elif free_leg_motion > 0.03: swing_score = 3
    elif free_leg_motion > 0.02: swing_score = 2
    else: swing_score = 1
    return swing_score

In [6]:
# Knee Flexion
def knee_score_cal(knee_val):
    knee_score = 0
    if knee_val < 90: knee_score = 5
    elif knee_val < 110: knee_score = 4
    elif knee_val < 130: knee_score = 3
    elif knee_val < 150: knee_score = 2
    else: knee_score = 1
    return knee_score

In [7]:
# Arm Movement
def arm_score_cal(arm_motion):
    arm_score = 0
    if arm_motion > 0.08: arm_score = 5
    elif arm_motion > 0.06: arm_score = 4
    elif arm_motion > 0.04: arm_score = 3
    elif arm_motion > 0.02: arm_score = 2
    else: arm_score = 1
    return arm_score

In [8]:
# Consecutive Hops
def hop_score_cal(hop_count):
    hop_score = 0
    if hop_count >= 5: hop_score = 5
    elif hop_count >= 4: hop_score = 4
    elif hop_count >= 3: hop_score = 3
    elif hop_count >= 2: hop_score = 2
    else: hop_score = 1
    return hop_score

In [9]:
def threshold_line(path):
    
    model = YOLO("best.pt")
    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])

        if len(left_x) > 50:
            break

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    print("Left cone X:", left_avg)
    print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    return left_avg, right_avg


In [14]:
# -------------------------------
# MAIN FUNCTION
# -------------------------------
def hopping_test(path="video.mp4"):

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    #threshold line calculate  
    left_line_x, right_line_x = 0.10, 0.85
    left_line_x, right_line_x = threshold_line(path)
    left_line_x = round((left_line_x/frame_width), 2) 
    right_line_x = round((right_line_x/frame_width), 2) 
    print(f"{left_line_x} --- {right_line_x}")

    l_ank_y, r_ank_y = [], []
    l_knee_ang, r_knee_ang = [], []
    l_el_y, r_el_y = [], []
    l_wri_y, r_wri_y = [], []
    hip_y = []

    frame_idx = 0
    window = "Hopping Analysis"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)

        if res.pose_landmarks:
            lms = res.pose_landmarks.landmark

            # hip center
            hx = (lms[23].x + lms[24].x) / 2

            # boundary check
            if not (left_line_x <= hx <= right_line_x):
                continue

            if lms[23].visibility < 0.5:
                cv2.imshow(window, frame)
                if cv2.waitKey(1) & 0xFF == 27: break
                continue

            mpDraw.draw_landmarks(frame, res.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAW_LM, DRAW_CONN)

            # points
            l_hip=(lms[23].x,lms[23].y); r_hip=(lms[24].x,lms[24].y)
            l_knee=(lms[25].x,lms[25].y); r_knee=(lms[26].x,lms[26].y)
            l_ank=(lms[27].x,lms[27].y); r_ank=(lms[28].x,lms[28].y)
            l_el=(lms[13].x,lms[13].y); r_el=(lms[14].x,lms[14].y)
            l_wri=(lms[15].x,lms[15].y); r_wri=(lms[16].x,lms[16].y)

            # signals
            l_ank_y.append(l_ank[1]); r_ank_y.append(r_ank[1])
            hip_y.append((l_hip[1]+r_hip[1])/2)

            l_knee_ang.append(angle(l_hip,l_knee,l_ank))
            r_knee_ang.append(angle(r_hip,r_knee,r_ank))

            l_el_y.append(l_el[1]); r_el_y.append(r_el[1])
            l_wri_y.append(l_wri[1]); r_wri_y.append(r_wri[1])

        cv2.imshow(window, frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    # -------------------------------
    # PREPROCESS
    # -------------------------------
    l_ank_y = smooth(l_ank_y)
    r_ank_y = smooth(r_ank_y)
    hip_y = smooth(hip_y)

    n = min(len(l_ank_y), len(r_ank_y), len(hip_y))
    if n < 12:
        return 1,1,1,1,1

    l_ank_y, r_ank_y, hip_y = l_ank_y[:n], r_ank_y[:n], hip_y[:n]

    # -------------------------------
    # STEP DETECTION (HOPS)
    # -------------------------------
    peaks,_ = find_peaks(-hip_y, distance=5)
    peaks = peaks[peaks < n]
    hop_count = len(peaks)

    # -------------------------------
    # SUPPORT FOOT DETECTION
    # -------------------------------
    support_seq = []
    for i in range(n):
        if l_ank_y[i] > r_ank_y[i]:
            support_seq.append("L")
        else:
            support_seq.append("R")

    # check consistency
    main_foot = max(set(support_seq), key=support_seq.count)
    same_count = support_seq.count(main_foot)

    # -------------------------------
    # FREE LEG SWING
    # measure variance of non-support leg
    # -------------------------------
    free_leg_motion = safe_std(l_ank_y - r_ank_y)

    # -------------------------------
    # KNEE FLEXION
    # -------------------------------
    knee_val = safe_mean(l_knee_ang + r_knee_ang)

    # -------------------------------
    # ARM MOVEMENT
    # wrist vertical oscillation
    # -------------------------------
    arm_motion = safe_std(l_wri_y) + safe_std(r_wri_y)

    # =========================================================
    # SCORING
    # =========================================================

    # Same Foot Landing
    foot_score = foot_score_cal(same_count)

    # Free Leg Swing
    swing_score = swing_score_cal(free_leg_motion)

    # Knee Flexion
    knee_score = knee_score_cal(knee_val)

    # Arm Movement
    arm_score = arm_score_cal(arm_motion)

    # Consecutive Hops
    hop_score = hop_score_cal(hop_count)

    print("------ HOPPING RESULT ------")
    print("Same Foot:", foot_score)
    print("Free Leg Swing:", swing_score)
    print("Knee Flexion:", knee_score)
    print("Arm Movement:", arm_score)
    print("Consecutive Hops:", hop_score)

    final_score = (foot_score + swing_score + knee_score + arm_score + hop_score)/5

    return final_score

In [15]:
path = "data/hopping.mp4"
print(hopping_test(path))

Left cone X: 97
Right cone X: 721
0.11 --- 0.85
------ HOPPING RESULT ------
Same Foot: 5
Free Leg Swing: 1
Knee Flexion: 1
Arm Movement: 4
Consecutive Hops: 5
3.2
